In [13]:
import os
from dotenv import load_dotenv
load_dotenv()
os.environ["GROQ_API_KEY"] = os.getenv("GROQ_API_KEY")
Weather_API_Key = os.getenv("WEATHER_API_KEY")

from langchain.chat_models import init_chat_model

llm = init_chat_model(model="openai/gpt-oss-120B", model_provider="groq")

In [14]:
from langchain.tools import tool
import requests
url = "https://api.openweathermap.org/data/2.5/weather?"
@tool
def get_weather(location:str)->str:
    """Take the location and return the current weather. """
    params={
        "q": location, 
        "appid" : Weather_API_Key, 
        "units" : "imperial"
    }

    try: 
        response = requests.get(url, params=params, timeout = 10)
        return response.json()
    except:
        return f"The requested weather information is unavailable."

In [15]:
get_weather.invoke("Chicago")

{'coord': {'lon': -87.65, 'lat': 41.85},
 'weather': [{'id': 800,
   'main': 'Clear',
   'description': 'clear sky',
   'icon': '01d'}],
 'base': 'stations',
 'main': {'temp': 97.21,
  'feels_like': 109.81,
  'temp_min': 94.55,
  'temp_max': 98.19,
  'pressure': 1012,
  'humidity': 51,
  'sea_level': 1012,
  'grnd_level': 991},
 'visibility': 10000,
 'wind': {'speed': 13.8, 'deg': 230, 'gust': 23.02},
 'clouds': {'all': 1},
 'dt': 1788382165,
 'sys': {'type': 2,
  'id': 2075214,
  'country': 'US',
  'sunrise': 1788347853,
  'sunset': 1788395006},
 'timezone': -18000,
 'id': 4887398,
 'name': 'Chicago',
 'cod': 200}

In [16]:
llm_with_tools = llm.bind_tools([get_weather])

In [17]:
from langchain.messages import HumanMessage, SystemMessage, AIMessage
messages = [
    SystemMessage(content="You are a helpful weather agent, take the location from the user query and use get_weather tool to generate the weather related query.")
]

In [18]:
user_input = input("What is your query?: ")
messages.append(HumanMessage(content=user_input))
aiMessage = llm_with_tools.invoke(messages)
messages.append(aiMessage)

#tool call
for tool_call in aiMessage.tool_calls:
    toolMessage = get_weather.invoke(tool_call)
    messages.append(toolMessage)
final_response = llm_with_tools.invoke(messages)
print(final_response.content)

**Current weather for Springfield, USA (as of the latest observation):**

- **Condition:** Broken clouds (about 60 % cloud cover)  
- **Temperature:** 97 °F (feels like 102 °F)  
- **Low / High today:** 95.9 °F / 99 °F  
- **Humidity:** 37 %  
- **Wind:** 8 mph from the southwest (200°)  
- **Visibility:** 10 km (clear)

It’s a hot day with mostly cloudy skies and low humidity, so it should feel quite warm outside. If you’re heading out, stay hydrated and consider a light jacket for the wind.


In [19]:
#generate the tool message
if aiMessage.tool_calls: 
    for tool_call in aiMessage.tool_calls:
        if(tool_call["name"] == get_weather.name):
            toolMessage = get_weather.invoke(tool_call)
            print(toolMessage)


content='{"coord": {"lon": -93.2982, "lat": 37.2153}, "weather": [{"id": 803, "main": "Clouds", "description": "broken clouds", "icon": "04d"}], "base": "stations", "main": {"temp": 97.48, "feels_like": 101.95, "temp_min": 95.9, "temp_max": 99, "pressure": 1013, "humidity": 37, "sea_level": 1013, "grnd_level": 968}, "visibility": 10000, "wind": {"speed": 8.05, "deg": 200}, "clouds": {"all": 59}, "dt": 1788382861, "sys": {"type": 2, "id": 2106595, "country": "US", "sunrise": 1788349488, "sunset": 1788396082}, "timezone": -18000, "id": 4409896, "name": "Springfield", "cod": 200}' name='get_weather' tool_call_id='fc_fa1243e4-98b9-4c76-b4dd-62e7af210a5a'


In [20]:
messages

[SystemMessage(content='You are a helpful weather agent, take the location from the user query and use get_weather tool to generate the weather related query.', additional_kwargs={}, response_metadata={}),
 HumanMessage(content='Weather in springfield today? ', additional_kwargs={}, response_metadata={}),
 AIMessage(content='', additional_kwargs={'reasoning_content': 'We need to extract location: "springfield". Use get_weather tool.', 'tool_calls': [{'id': 'fc_fa1243e4-98b9-4c76-b4dd-62e7af210a5a', 'function': {'arguments': '{"location":"Springfield"}', 'name': 'get_weather'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 44, 'prompt_tokens': 158, 'total_tokens': 202, 'completion_time': 0.091374108, 'completion_tokens_details': {'reasoning_tokens': 16}, 'prompt_time': 0.006287403, 'prompt_tokens_details': None, 'queue_time': 0.091287827, 'total_time': 0.097661511}, 'model_name': 'openai/gpt-oss-120B', 'system_fingerprint': 'fp_90620edd96', 'service_tier

In [21]:
response = llm.invoke(messages)
print(response.content)

**Current weather for Springfield, MO (USA) – September 2 2026**

| Element | Details |
|---------|---------|
| **Condition** | Broken clouds (≈ 60 % cloud cover) |
| **Temperature** | 97 °F (feels like 102 °F) |
| **High / Low (today)** | Around 99 °F / 96 °F |
| **Humidity** | 37 % |
| **Wind** | 8 mph from the south‑south‑west (200°) |
| **Visibility** | 10 km (clear) |
| **Pressure** | 1013 hPa (sea‑level) |
| **Sunrise / Sunset** | 6:31 am / 7:41 pm (local CDT, UTC‑5) |
| **Local time of observation** | 12:41 pm (mid‑day) |

**What this means**

- It’s a hot, sunny‑ish day with plenty of sunshine breaking through the clouds.  
- The “feels‑like” temperature is a few degrees higher than the actual reading, so stay hydrated and consider shade or air‑conditioning if you’ll be outdoors.  
- Light to moderate wind from the south‑south‑west will provide a gentle breeze, but it won’t be enough to offset the heat.  
- Humidity is relatively low, which helps the heat feel a bit more tolera